# WarehouseSort × SmolVLA — Colab T4 재현본

원본 30K SmolVLA 평가와 **System 2/1 hybrid**를 Colab T4에서 재현합니다.

- System 2: 30K VLA가 RGB + robot state + task prompt로 target/retry를 판단
- System 1: 코드 controller가 축 고정 Cartesian 이동·grasp·release를 실행
- 서버 검증값: Hard seed 42/43/44 각각 2/6, 총 6/18 (33.33%)
- 기본 설정은 순수 VLA 평가를 건너뛰고 Hard와 Medium hybrid를 실행합니다.


## 0. Colab 설정

`런타임 → 런타임 유형 변경 → T4 GPU`를 선택한 다음 아래 셀부터 실행하세요.
GitHub에 체크포인트를 올린 뒤 `CHECKPOINT_REPO`와 `CHECKPOINT_SUBDIR`만 수정하면 됩니다.
1.2GB `model.safetensors`는 일반 Git 파일이 아니라 **Git LFS**로 올려야 합니다.


In [ ]:
# ===== 실행 설정 =====
CHECKPOINT_REPO = "https://github.com/Ahnseongmin1749/Marso-Hack-Berlin-2026.git"
CHECKPOINT_SUBDIR = "checkpoints/smolvla_all_recovery_30k"

SEEDS = [42, 43, 44]

# 모든 rollout은 반드시 System 2(SmolVLA) + System 1(code controller)을 사용합니다.
RUN_HYBRID = True
HYBRID_LEVELS = ["hard", "medium"]
HYBRID_SEEDS = [42, 43, 44]  # 빠른 확인은 [42]
HYBRID_RECORD_VIDEO = True

# 순수 VLA 영상 설정
RECORD_VIDEO = True
VIDEO_SEEDS = [SEEDS[0]]

EXPECTED_TRANSFORMERS = None
EXPECTED_HF_HUB = None

assert CHECKPOINT_REPO.endswith(".git")


## 1. 패키지 설치

처음 한 번 실행한 뒤 **런타임 → 세션 다시 시작**을 누르세요. 재시작 후에는 **0번 설정 셀을 다시 실행**하고, 설치 셀은 건너뛴 뒤 2번부터 계속하면 됩니다.


In [ ]:
%pip uninstall -y -q gradio gradio-client transformers huggingface-hub lerobot
%pip install -q --no-cache-dir --force-reinstall \
    torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 \
    --index-url https://download.pytorch.org/whl/cu128
%pip install -q --no-cache-dir \
    mani-skill==3.0.1 gymnasium==1.3.0 hydra-core==1.3.3 imageio-ffmpeg opencv-python-headless
%pip install -q --no-cache-dir "lerobot[smolvla]==0.4.4"
!apt-get -qq update && apt-get -qq install -y git-lfs
!git lfs install
print("설치 완료 — 런타임을 다시 시작한 뒤 2번부터 실행하세요.")


## 2. GPU 및 라이브러리 확인

서버와 Colab의 버전 차이가 재현성에 영향을 줄 수 있으므로 핵심 패키지 버전을 함께 출력합니다.


In [3]:
from importlib.metadata import version, PackageNotFoundError
import torch


def safe_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "NOT INSTALLED"

versions = {
    "torch": torch.__version__,
    "torchvision": safe_version("torchvision"),
    "lerobot": safe_version("lerobot"),
    "transformers": safe_version("transformers"),
    "huggingface-hub": safe_version("huggingface-hub"),
    "gymnasium": safe_version("gymnasium"),
    "mani-skill": safe_version("mani-skill"),
    "numpy": safe_version("numpy"),
}

print("=== Colab environment fingerprint ===")
for k, v in versions.items():
    print(f"{k:16s}: {v}")

print("CUDA available   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU              :", torch.cuda.get_device_name(0))
    print("VRAM(GB)         :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
    print("CUDA runtime     :", torch.version.cuda)

assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
assert versions["lerobot"] == "0.4.4", f"LeRobot 버전 불일치: {versions['lerobot']}"

if EXPECTED_TRANSFORMERS is not None:
    assert versions["transformers"] == EXPECTED_TRANSFORMERS, (
        f"transformers 버전 불일치: Colab={versions['transformers']} / server={EXPECTED_TRANSFORMERS}"
    )
if EXPECTED_HF_HUB is not None:
    assert versions["huggingface-hub"] == EXPECTED_HF_HUB, (
        f"huggingface-hub 버전 불일치: Colab={versions['huggingface-hub']} / server={EXPECTED_HF_HUB}"
    )

print()
print("✅ 기본 환경 확인 완료")


=== Colab environment fingerprint ===
torch           : 2.10.0+cu128
torchvision     : 0.25.0+cu128
lerobot         : 0.4.4
transformers    : 4.57.6
huggingface-hub : 0.35.3
gymnasium       : 1.3.0
mani-skill      : 3.0.1
numpy           : 2.2.6
CUDA available   : True
GPU              : Tesla T4
VRAM(GB)         : 14.6
CUDA runtime     : 12.8

✅ 기본 환경 확인 완료


## 3. WarehouseSort 코드 준비

평가 당시 사용한 대회 저장소 commit으로 고정합니다.


In [4]:
from pathlib import Path
import os, sys, subprocess

REPO = Path("/content/berlin-marso-hackathon")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/marso-robotics/berlin-marso-hackathon.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "6048f33217f26ae39009a812f53c81171517f393"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "--no-deps"], check=True)
os.chdir(REPO)

os.environ["DISPLAY"] = ""
os.environ["PYOPENGL_PLATFORM"] = "egl"
sys.path.insert(0, str(REPO))
import warehouse_sort
print("WarehouseSort 준비 완료")


/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:42: UserWarning: Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.
  warn("Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.")
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(


WarehouseSort 준비 완료


## 4. GitHub에서 체크포인트 다운로드

private repository라면 clone 전에 Colab secret 또는 Git credential 설정이 필요합니다.
Git LFS가 `model.safetensors`의 실제 내용을 받았는지도 다음 셀에서 검사합니다.


In [5]:
from pathlib import Path
import subprocess

CKPT_REPO_DIR = Path("/content/smolvla-checkpoint-repo")

if not CKPT_REPO_DIR.exists():
    subprocess.run(["git", "clone", CHECKPOINT_REPO, str(CKPT_REPO_DIR)], check=True)

# 해당 checkpoint 하위의 LFS 파일만 우선 pull합니다.
include_pattern = f"{CHECKPOINT_SUBDIR}/**"
subprocess.run(
    ["git", "-C", str(CKPT_REPO_DIR), "lfs", "pull", "--include", include_pattern],
    check=True,
)

SMOLVLA_CKPT = CKPT_REPO_DIR / CHECKPOINT_SUBDIR

# LeRobot checkpoint 전체 폴더를 올린 경우 pretrained_model/ 한 단계 안쪽을 자동 탐색
if not (SMOLVLA_CKPT / "model.safetensors").exists():
    nested = SMOLVLA_CKPT / "pretrained_model"
    if (nested / "model.safetensors").exists():
        SMOLVLA_CKPT = nested

print("checkpoint:", SMOLVLA_CKPT)


checkpoint: /content/smolvla-checkpoint-repo/checkpoints/smolvla_all_recovery_30k


## 5. 체크포인트 무결성 확인

추론에는 optimizer/training state가 필요 없습니다. 모델 weight와 저장된 pre/postprocessor를 확인하고, Git LFS pointer 파일을 실제 weight로 착각하지 않도록 검사합니다.


In [6]:
required = [
    "config.json",
    "model.safetensors",
    "policy_preprocessor.json",
    "policy_preprocessor_step_5_normalizer_processor.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor_step_0_unnormalizer_processor.safetensors",
]

for name in required:
    p = SMOLVLA_CKPT / name
    assert p.exists(), f"누락 파일: {p}"
    print(f"{name:70s} {p.stat().st_size / 1024**2:9.2f} MiB")

model_file = SMOLVLA_CKPT / "model.safetensors"

# Git LFS pointer는 보통 매우 작은 텍스트 파일입니다.
assert model_file.stat().st_size > 10_000_000, (
    "model.safetensors가 비정상적으로 작습니다. Git LFS 실제 weight가 아니라 pointer일 수 있습니다."
)
with open(model_file, "rb") as f:
    header = f.read(256)
assert b"git-lfs.github.com/spec" not in header, (
    "model.safetensors가 Git LFS pointer입니다. git lfs pull을 확인하세요."
)

print()
print(f"✅ model.safetensors 실제 크기: {model_file.stat().st_size / 1024**3:.2f} GiB")
print("✅ 체크포인트 무결성 확인 완료")


config.json                                                                 0.00 MiB
model.safetensors                                                        1142.30 MiB
policy_preprocessor.json                                                    0.00 MiB
policy_preprocessor_step_5_normalizer_processor.safetensors                 0.01 MiB
policy_postprocessor.json                                                   0.00 MiB
policy_postprocessor_step_0_unnormalizer_processor.safetensors              0.01 MiB

✅ model.safetensors 실제 크기: 1.12 GiB
✅ 체크포인트 무결성 확인 완료


## 6. SmolVLA 로드

체크포인트에 저장된 normalization과 unnormalization processor도 함께 로드합니다.


In [7]:
import gc
import torch
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

# 이전 실행 잔여 메모리 정리
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda")
policy = SmolVLAPolicy.from_pretrained(str(SMOLVLA_CKPT)).to(device).eval()
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config,
    pretrained_path=str(SMOLVLA_CKPT),
    preprocessor_overrides={"device_processor": {"device": "cuda"}},
)

print("SmolVLA 로드 완료")
print("chunk_size     :", policy.config.chunk_size)
print("n_action_steps :", policy.config.n_action_steps)
print("device         :", policy.config.device)

# 서버 학습 설정과 checkpoint가 맞는지 즉시 확인
assert policy.config.chunk_size == 16, f"예상 chunk_size=16, 실제={policy.config.chunk_size}"
assert policy.config.n_action_steps == 1, f"예상 n_action_steps=1, 실제={policy.config.n_action_steps}"

policy.reset()
preprocessor.reset()
postprocessor.reset()
print("✅ 서버 학습 설정과 checkpoint config 일치")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

processor_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/868 [00:00<?, ?B/s]

Reducing the number of VLM layers to 16 ...
Loading weights from local directory
SmolVLA 로드 완료
chunk_size     : 16
n_action_steps : 1
device         : cuda
✅ 서버 학습 설정과 checkpoint config 일치


## 7. 순수 VLA 점수 평가 — 비활성화

순수 VLA action을 환경에 직접 넣는 경로는 사용하지 않습니다. 아래 System 2/1 hybrid 평가를 실행하세요.


In [ ]:
print("순수 VLA action을 환경에 직접 넣는 경로는 사용하지 않습니다. 아래 System 2/1 hybrid 평가를 실행하세요.")


## 8. 순수 VLA 점수 집계 — 비활성화

점수는 hybrid 평가 셀에서만 집계합니다.


In [ ]:
print("점수는 hybrid 평가 셀에서만 집계합니다.")


## 9. 순수 VLA 영상 생성 — 비활성화

영상도 hybrid controller가 실행된 rollout만 생성합니다.


In [ ]:
print("영상도 hybrid controller가 실행된 rollout만 생성합니다.")


## 10. 결과 다운로드

In [11]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/smolvla_eval", "zip", RESULT_ROOT)
print(archive)
# 필요할 때 다음 줄의 주석을 해제하세요.
# files.download(archive)


/content/smolvla_eval.zip


## 11. System 2/1 hybrid — Hard 평가

30K VLA는 target 선택 및 실패 후 retry/skip 판단만 합니다. 실제 action은 axis-locked controller가 실행하므로 target을 잡은 뒤 불필요하게 되돌아가는 궤적을 줄입니다. 구현은 체크포인트와 같은 GitHub 저장소의 `hybrid/`에서 불러옵니다.


In [ ]:
import json, random, sys
import numpy as np
from pathlib import Path
from lerobot.policies.utils import prepare_observation_for_inference
from warehouse_sort.utils import compose_cfg, make_env

sys.path.insert(0, str(CKPT_REPO_DIR))
from hybrid.rgb_axis_locked_policy import RGBParcelDetector
from hybrid.vla_advisory_hybrid import VLAAdvisoryHybrid

HYBRID_ROOT = Path("/content/vla_advisory_hybrid_eval")
TASK = "Sort each parcel into the bin that matches the color of the tag on top of the parcel."

def hybrid_advisor(frame_rgb, state):
    frame = prepare_observation_for_inference(
        {
            "observation.images.scene": frame_rgb,
            "observation.state": state.detach().cpu().numpy().astype(np.float32),
        },
        device=device, task=TASK, robot_type="franka_panda",
    )
    with torch.inference_mode():
        return postprocessor(policy.select_action(preprocessor(frame)))[..., :4]

def set_hybrid_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

hybrid_summary = []
if RUN_HYBRID:
    HYBRID_ROOT.mkdir(parents=True, exist_ok=True)
    for level in HYBRID_LEVELS:
        cfg = compose_cfg([f"difficulty={level}"])
        for seed in HYBRID_SEEDS:
            set_hybrid_seeds(seed)
            video_dir = HYBRID_ROOT / level / f"seed_{seed}" if HYBRID_RECORD_VIDEO else None
            env = None
            try:
                env, _ = make_env(cfg, "rgb", cfg.randomization, num_envs=1,
                                  video_dir=str(video_dir) if video_dir else None)
                obs, _ = env.reset(seed=[seed])
                params = env.unwrapped._sensors["scene_camera"].get_params()
                detector = RGBParcelDetector(
                    params["intrinsic_cv"][0].detach().cpu().numpy(),
                    params["extrinsic_cv"][0].detach().cpu().numpy(),
                )
                policy.reset(); preprocessor.reset(); postprocessor.reset()
                hybrid = VLAAdvisoryHybrid(advisor=hybrid_advisor, detector=detector)
                hybrid.reset(obs["rgb"][0].detach().cpu().numpy())
                choices, max_sorted = [], 0.0

                for _ in range(int(cfg.max_episode_steps) - 1):
                    rgb, state = obs["rgb"][0].detach().cpu().numpy(), obs["state"][0]
                    if hybrid.current is None and hybrid.remaining:
                        choice = hybrid.choose(rgb, state)
                        choices.append({"color": int(choice.source.color),
                                        "pixel": list(choice.source.pixel),
                                        "score": choice.score,
                                        "confidence": choice.confidence,
                                        "advice": list(choice.advice)})
                    action = torch.zeros(4, device=state.device) if hybrid.done else hybrid.act(rgb, state)
                    obs, _, _, _, info = env.step(action.unsqueeze(0))
                    if "success_count" in info:
                        max_sorted = max(max_sorted, float(info["success_count"][0].item()))

                final_sorted = float(env.unwrapped.evaluate()["success_count"][0].item())
                row = {
                    "level": level, "seed": seed, "sorted": max(max_sorted, final_sorted),
                    "total": int(cfg.difficulty.num_parcels), "choices": choices,
                    "system2_events": [
                        {"primitive": e.primitive, "retry": e.retry, "score": e.score,
                         "advice": list(e.advice)} for e in hybrid.events
                    ],
                    "video_dir": str(video_dir) if video_dir else None,
                }
                hybrid_summary.append(row)
                print(f"{level} seed={seed}: {row['sorted']:.0f}/{row['total']} "
                      f"choices={len(choices)} events={len(hybrid.events)}")
            finally:
                if env is not None: env.close()
                torch.cuda.empty_cache()

    (HYBRID_ROOT / "summary.json").write_text(
        json.dumps(hybrid_summary, ensure_ascii=False, indent=2), encoding="utf-8")
    total = sum(x["sorted"] for x in hybrid_summary)
    denom = sum(x["total"] for x in hybrid_summary)
    print(f"HYBRID TOTAL: {total:.0f}/{denom} = {total / denom:.2%}" if denom else "결과 없음")
else:
    print("RUN_HYBRID=False — hybrid 평가를 건너뜁니다.")


## 12. Hybrid 영상 재생

환경이 닫힌 뒤 생성된 MP4를 찾아 재생합니다.


In [ ]:
from IPython.display import Video, display
import ipywidgets as widgets

hybrid_videos = []
for row in hybrid_summary:
    if row["video_dir"]:
        found = sorted(Path(row["video_dir"]).rglob("*.mp4"))
        if found:
            hybrid_videos.append((f"{row['level']} seed {row['seed']} — {row['sorted']:.0f}/{row['total']}", str(found[0])))

if hybrid_videos:
    picker = widgets.Dropdown(options=hybrid_videos, description="hybrid:")
    output = widgets.Output()
    def show_hybrid_video(change=None):
        with output:
            output.clear_output(wait=True)
            display(Video(picker.value, embed=True, width=800))
    picker.observe(show_hybrid_video, names="value")
    display(picker, output); show_hybrid_video()
else:
    print("생성된 hybrid MP4가 없습니다. HYBRID_RECORD_VIDEO=True인지 확인하세요.")


## 13. Hybrid 결과 다운로드

In [ ]:
import shutil
if HYBRID_ROOT.exists():
    hybrid_archive = shutil.make_archive("/content/vla_advisory_hybrid_eval", "zip", HYBRID_ROOT)
    print(hybrid_archive)
    # 필요할 때 주석 해제:
    # from google.colab import files
    # files.download(hybrid_archive)


## 14. 학습 부분 — T4 추론본에서는 비활성화

아래 코드는 실행 기록과 설정 참고용입니다. 이 Colab 노트북에서는 데이터 600 episodes 변환,
recovery 수집, 30K 학습을 실행하지 않습니다. 체크포인트는 GitHub에서 가져옵니다.


In [ ]:
# 실행하지 않음: 대용량 Kaggle 데이터 다운로드 및 LeRobot 변환
# !python prepare_multilevel_recovery_dataset.py

# 실행하지 않음: V100에서 수행한 원본 30K action-expert 학습 설정
# !python -m lerobot.scripts.lerobot_train \
#   --dataset.repo_id=local/warehouse_smolvla_all_recovery \
#   --dataset.root=/content/warehouse_smolvla_all_recovery \
#   --policy.type=smolvla \
#   --policy.pretrained_path=lerobot/smolvla_base \
#   --policy.train_expert_only=true \
#   --policy.freeze_vision_encoder=true \
#   --policy.chunk_size=16 \
#   --policy.n_action_steps=1 \
#   --batch_size=8 --steps=30000 --optimizer.lr=1e-4

print("학습 셀은 의도적으로 주석 처리되어 있습니다.")


## 재현 기준

- 동일 checkpoint
- 동일 WarehouseSort commit
- 동일 seed
- 가능한 한 동일한 Python package 버전
- 점수 평가는 영상 녹화 없이 수행
- 영상은 별도 rollout으로 생성

기존 서버 결과는 Easy 6/6, Medium 8/12, Hard 1/18 (seeds 42, 43, 44), weighted 42.78%입니다. GPU 연산의 미세한 비결정성 때문에 궤적이 byte-identical하지는 않을 수 있습니다.

System 2/1 hybrid의 서버 기준 Hard 결과는 seed 42/43/44 각각 2/6, 총 6/18입니다.
